In [ ]:
import ccxt
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

def get_cs(exch, symbol, tf="5m", from_ts=None, to_ts=None):
    buf = [[0, 0, 0, 0, 0, 0]]

    while buf[-1][0] < to_ts:
        time.sleep(5)
        try:
            res = exch.fetchOHLCV(symbol, tf, since=from_ts, limit=1000)
        except Exception as e:
            time.sleep(5)
            res = exch.fetchOHLCV(symbol, tf, since=from_ts, limit=1000)
        buf.extend(res)
        from_ts = res[-1][0]
    return buf

# symbols = ["ETH", "SOL", "XRP", "SUI", "PEPE"]
symbols = ["PEPE"]
ts_start = 1763078400000
ts_end = 1764952500000

for symbol in tqdm(symbols):

    tpe = ThreadPoolExecutor(max_workers=5)

    bg, bg_perp, gate, gate_perp, kucoin = ccxt.bitget(), ccxt.bitget(), ccxt.gate(), ccxt.gate(), ccxt.kucoin()
    res = tpe.map(
        get_cs,
        [bg, bg_perp, gate, gate_perp, kucoin],
        [f"{symbol}/USDT", f"{symbol}/USDT:USDT", f"{symbol}/USDT", f"{symbol}/USDT:USDT", f"{symbol}/USDT"],
        ["5m"]*5,
        [ts_start]*5,
        [ts_end]*5,
    )
    res_obj = list(res)

    for i, name in zip(res_obj, ["bg", "bg_perp", "gate", "gate_perp", "kucoin"]):
        df_i = pd.DataFrame(i, columns=["ts", "open", "high", "low", "close", "volume"])
        df_i["exch"] = name
        df_i["symbol"] = symbol
        df_i.to_csv(f"ohlcv_data/{name}_{symbol}USDT_5m_ts_14nov_5dec.csv", index=False)


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:51<00:00, 51.76s/it]


agg

In [ ]:
import os

df = pd.concat([pd.read_csv(f"ohlcv_data/{f}") for f in os.listdir("ohlcv_data")], axis=0, ignore_index=True)
df = df[df.ts != 0].drop_duplicates(subset=["ts", "symbol", "exch"]).reset_index(drop=True).copy()
df = df[(df.ts > 1763078400000) & (df.ts < 1764963600000)].reset_index(drop=True).copy()
df.to_csv("ohlcv_14nov_5dec_agg.csv", index=False)

In [ ]:
import ccxt

bg, bg_perp, gate, gate_perp, kucoin = ccxt.bitget(), ccxt.bitget(), ccxt.gate(), ccxt.gate(), ccxt.kucoin()

bg_perp.fetchOHLCV("ETH/USDT:USDT", "5m", limit=1000)

In [ ]:
bg_perp.fetchOHLCV("ETH/USDT:USDT", "5m", limit=1000)

In [12]:
from datetime import datetime

datetime.fromtimestamp(1765127100000 / 1000)

datetime.datetime(2025, 12, 7, 17, 5)